In [1]:
import os
import json
import time
from openai import OpenAI
import pickle
import numpy as np
from collections import Counter

In [ ]:
print("\n[1/4] Loading Day 2 explanations...")

if not os.path.exists('../results/day2_explanations.pkl'):
    print("[ERROR] day2_explanations.pkl not found!")
    print("Please complete Day 2 first.")
else:
    with open('../results/day2_explanations.pkl', 'rb') as f:
        explanations = pickle.load(f)
    
    print(f"[OK] Loaded {len(explanations)} explanations from Day 2")
    
    if len(explanations) == 0:
        print("\n[WARNING] No explanations found!")
        print("You need to fix Day 2 first before proceeding to Day 3.")
        print("Day 3 requires GraphLIME explanations as input.")
    else:
        print(f"[OK] Ready to generate LLM explanations")


[1/4] Loading Day 2 explanations...
✓ Loaded 50 explanations from Day 2
✓ Ready to generate LLM explanations


In [ ]:
from dotenv import load_dotenv
load_dotenv()
api_key = os.environ.get('OPENAI_API_KEY')

if not api_key:
    print("key not found")
else:
    client = OpenAI(api_key=api_key)
    print("[OK] OpenAI client initialized")
print("\n[3/4] Defining fraud taxonomy...")


FRAUD_TYPES = [
    "Ponzi schemes",
    "Phishing attacks", 
    "Pump-and-dump schemes",
    "Ransomware",
    "Money laundering",
    "Giveaway scams",
    "Impersonation scams",
    "Securities fraud",
    "Mixing services",
    "Dark web marketplace"
]

for i, fraud_type in enumerate(FRAUD_TYPES):
    print(f"{i}. {fraud_type}")

✓ OpenAI client initialized

[3/4] Defining fraud taxonomy...
0. Ponzi schemes
1. Phishing attacks
2. Pump-and-dump schemes
3. Ransomware
4. Money laundering
5. Giveaway scams
6. Impersonation scams
7. Securities fraud
8. Mixing services
9. Dark web marketplace


In [ ]:
def create_llm_prompt(explanation_data, fraud_types=FRAUD_TYPES):
    """
    Create a well-structured prompt for the LLM
    
    This prompt:
    1. Provides context (crypto forensics)
    2. Gives the data (features + importance)
    3. Specifies output format (strict JSON)
    4. Lists allowed fraud types
    """
    
    # Format GraphLIME feature importance
    formatted_features = "GraphLIME Feature Importance (what the model focused on):\n"
    for feat_idx, weight in explanation_data['top_features']:
        feat_value = explanation_data['node_features'][feat_idx]
        formatted_features += f"  • Feature {feat_idx}: importance={weight:.4f}, value={feat_value:.4f}\n"
    
    # Create the prompt
    prompt = f"""You are a cryptocurrency fraud analyst reviewing a suspicious Bitcoin wallet.

**Wallet Information:**
- Wallet ID: {explanation_data['node_id']}
- Anomaly Score: {explanation_data['anomaly_score']:.4f} (lower scores = more suspicious)

**Model's Reasoning (GraphLIME Analysis):**
{formatted_features}

**Your Task:**
Analyze this wallet and determine if it shows signs of fraud. Consider:
- Transaction patterns indicated by the features
- Typical behaviors associated with different fraud types
- The model's confidence (shown by feature importance weights)

**Response Format (STRICT JSON ONLY):**
Return ONLY valid JSON with this exact structure:

{{
  "explanation": "2-5 sentences explaining why this wallet is or isn't suspicious",
  "is_fraud": true or false,
  "fraud_type": "one of the types below, or null if not fraud",
  "confidence": 0.0 to 1.0,
  "evidence": {{
    "features": ["list of feature numbers that support your conclusion"],
    "behaviors": ["list of specific suspicious patterns observed"]
  }}
}}

**Allowed Fraud Types:**
{', '.join(fraud_types)}

**Rules:**
- If not fraudulent, set fraud_type to null
- Base your analysis on the feature values and importance weights
- Be specific about which features led to your conclusion
- Confidence should reflect how clear the evidence is
- Return ONLY the JSON object, no markdown formatting, no preamble
"""
    
    return prompt


def get_llm_explanation(explanation_data, client, model="gpt-4o-mini", temperature=0.2):
    """
    Get LLM explanation for a single node
    
    Returns:
        result: Dictionary with explanation
        usage: Dictionary with token counts and cost
    """
    
    prompt = create_llm_prompt(explanation_data)
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system", 
                    "content": "You are a cryptocurrency fraud detection expert. Analyze transaction patterns and return only JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=temperature,
            max_tokens=500
        )
        
        # Extract response
        content = response.choices[0].message.content.strip()
        
        # Remove markdown code blocks if present
        if content.startswith("```"):
            # Remove ```json and ``` markers
            lines = content.split('\n')
            content = '\n'.join(lines[1:-1]) if len(lines) > 2 else content
            content = content.replace("```json", "").replace("```", "").strip()
        
        # Parse JSON
        result = json.loads(content)
        
        # Calculate cost (GPT-4o-mini pricing as of Dec 2024)
        # Input: $0.150 per 1M tokens, Output: $0.600 per 1M tokens
        cost = (response.usage.prompt_tokens * 0.150 / 1_000_000 + 
                response.usage.completion_tokens * 0.600 / 1_000_000)
        
        usage = {
            'prompt_tokens': response.usage.prompt_tokens,
            'completion_tokens': response.usage.completion_tokens,
            'total_tokens': response.usage.total_tokens,
            'cost': cost
        }
        
        return result, usage
        
    except json.JSONDecodeError as e:
        print(f"    [WARNING] JSON parse error: {e}")
        print(f"    Response was: {content[:100]}...")
        return None, None
        
    except Exception as e:
        print(f"    [WARNING] API error: {e}")
        return None, None

In [5]:
def get_llm_consensus(explanation_data, client, n_samples=5):
    """
    Poll LLM multiple times and build consensus
    
    Why? LLMs can be inconsistent. By asking 5 times and 
    taking the majority vote, we get more reliable results.
    
    Args:
        explanation_data: Dictionary with node info from Day 2
        client: OpenAI client
        n_samples: Number of times to poll LLM (default 5)
        
    Returns:
        Dictionary with samples, consensus, and usage stats
    """
    
    samples = []
    total_cost = 0.0
    total_tokens = 0
    
    for sample_num in range(n_samples):
        result, usage = get_llm_explanation(explanation_data, client)
        
        if result is not None:
            samples.append(result)
            total_cost += usage['cost']
            total_tokens += usage['total_tokens']
        
        # Small delay to avoid rate limits
        time.sleep(1)
    
    # Build consensus from samples
    if len(samples) == 0:
        return None
    
    # Majority vote for is_fraud
    fraud_votes = [s['is_fraud'] for s in samples]
    is_fraud_consensus = sum(fraud_votes) > len(fraud_votes) / 2
    
    # Get fraud type from fraud samples only
    fraud_samples = [s for s in samples if s['is_fraud']]
    if fraud_samples:
        fraud_types = [s['fraud_type'] for s in fraud_samples if s['fraud_type']]
        if fraud_types:
            # Most common fraud type
            fraud_type_consensus = max(set(fraud_types), key=fraud_types.count)
            fraud_type_agreement = fraud_types.count(fraud_type_consensus) / len(fraud_types)
        else:
            fraud_type_consensus = None
            fraud_type_agreement = None
    else:
        fraud_type_consensus = None
        fraud_type_agreement = None
    
    # Average confidence
    avg_confidence = np.mean([s['confidence'] for s in samples])
    
    # Agreement rate (how many agreed with consensus)
    agreement_rate = sum(v == is_fraud_consensus for v in fraud_votes) / len(fraud_votes)
    
    consensus = {
        'node_id': explanation_data['node_id'],
        'anomaly_score': explanation_data['anomaly_score'],
        'is_fraud': is_fraud_consensus,
        'fraud_type': fraud_type_consensus,
        'avg_confidence': float(avg_confidence),
        'agreement_rate': float(agreement_rate),
        'fraud_type_agreement': float(fraud_type_agreement) if fraud_type_agreement else None,
        'samples': samples,
        'usage': {
            'total_cost': total_cost,
            'total_tokens': total_tokens,
            'n_samples': len(samples)
        }
    }
    
    return consensus

In [ ]:
print("\n" + "="*70)
print("GENERATING LLM EXPLANATIONS")
print("="*70)

# Check we have explanations and API key
if len(explanations) == 0:
    print("\n[WARNING] No Day 2 explanations found!")
    print("Cannot proceed with Day 3.")
    print("Please fix Day 2 first.")
else:
    if 'client' not in locals():
        print("\n[WARNING] OpenAI client not initialized!")
        print("Please set your API key in Cell 1 and re-run.")
    else:
        print(f"\nConfiguration:")
        print(f"  Nodes to process: {len(explanations)}")
        print(f"  Samples per node: 5")
        print(f"  Total LLM calls: {len(explanations) * 5}")
        print(f"  Estimated cost: ${len(explanations) * 5 * 0.001:.2f}")
        print(f"  Estimated time: ~{len(explanations) * 5 * 2 / 60:.0f} minutes")
        print(f"\n[WARNING] This will use your OpenAI API credits!")
        
        # Ask for confirmation
        print("\n" + "─"*70)
        user_input = input("Type 'yes' to proceed with LLM generation: ")
        
        if user_input.lower() != 'yes':
            print("\n[CANCELLED] Cancelled by user")
        else:
            print("\n[START] Starting LLM generation...\n")
            
            llm_results = {}
            total_cost = 0.0
            failed_count = 0
            start_time = time.time()
            
            # Process each explanation
            for i, (idx, explanation_data) in enumerate(explanations.items(), 1):
                try:
                    # Get consensus from 5 LLM samples
                    consensus = get_llm_consensus(explanation_data, client, n_samples=5)
                    
                    if consensus is not None:
                        llm_results[idx] = consensus
                        total_cost += consensus['usage']['total_cost']
                        
                        # Show progress
                        if i % 10 == 0 or i == len(explanations):
                            elapsed = time.time() - start_time
                            avg_time = elapsed / i
                            eta = avg_time * (len(explanations) - i)
                            
                            print(f"Progress: {i}/{len(explanations)} | "
                                  f"Cost: ${total_cost:.3f} | "
                                  f"Elapsed: {elapsed/60:.1f}m | "
                                  f"ETA: {eta/60:.1f}m | "
                                  f"Failed: {failed_count}")
                    else:
                        failed_count += 1
                        if failed_count <= 3:
                            print(f"  [WARNING] Failed on node {idx}")
                
                except KeyboardInterrupt:
                    print("\n\n[WARNING] Interrupted by user!")
                    print(f"Processed {len(llm_results)}/{len(explanations)} nodes")
                    break
                    
                except Exception as e:
                    failed_count += 1
                    if failed_count <= 3:
                        print(f"  [WARNING] Error on node {idx}: {str(e)[:50]}")
                    continue
            
            elapsed_time = time.time() - start_time
            
            print("\n" + "="*70)
            print("LLM GENERATION COMPLETE")
            print("="*70)
            
            print(f"\n📊 Results:")
            print(f"  Attempted: {len(explanations)}")
            print(f"  Successful: {len(llm_results)}")
            print(f"  Failed: {failed_count}")
            print(f"  Success rate: {len(llm_results)/len(explanations)*100:.1f}%")
            print(f"\n💰 Cost:")
            print(f"  Total: ${total_cost:.3f}")
            if len(llm_results) > 0:
                print(f"  Per node: ${total_cost/len(llm_results):.4f}")
            print(f"\n⏱️  Time:")
            print(f"  Total: {elapsed_time/60:.1f} minutes")

            if len(llm_results) > 0:                print(f"  Per node: {elapsed_time/len(llm_results):.1f} seconds")


GENERATING LLM EXPLANATIONS

Configuration:
  Nodes to process: 50
  Samples per node: 5
  Total LLM calls: 250
  Estimated cost: $0.25
  Estimated time: ~8 minutes

⚠️  This will use your OpenAI API credits!

──────────────────────────────────────────────────────────────────────

 Starting LLM generation...

Progress: 10/50 | Cost: $0.007 | Elapsed: 3.1m | ETA: 12.2m | Failed: 0
Progress: 20/50 | Cost: $0.014 | Elapsed: 6.0m | ETA: 9.0m | Failed: 0
Progress: 30/50 | Cost: $0.022 | Elapsed: 9.2m | ETA: 6.1m | Failed: 0
Progress: 40/50 | Cost: $0.029 | Elapsed: 12.4m | ETA: 3.1m | Failed: 0
Progress: 50/50 | Cost: $0.036 | Elapsed: 15.4m | ETA: 0.0m | Failed: 0

LLM GENERATION COMPLETE

📊 Results:
  Attempted: 50
  Successful: 50
  Failed: 0
  Success rate: 100.0%

💰 Cost:
  Total: $0.036
  Per node: $0.0007

⏱️  Time:
  Total: 15.4 minutes
  Per node: 18.5 seconds


In [ ]:
if len(llm_results) > 0:
    print("\n" + "="*70)
    print("SAVING RESULTS")
    print("="*70)
    
    # Save results
    output_path = '../results/day3_llm_results.pkl'
    with open(output_path, 'wb') as f:
        pickle.dump(llm_results, f)
    
    print(f"\n[SAVED] {len(llm_results)} LLM results to:")
    print(f"  {output_path}")
    print(f"  File size: {os.path.getsize(output_path) / 1024:.1f} KB")
    
    # Analysis
    print("\n" + "="*70)
    print("ANALYSIS")
    print("="*70)
    
    # Fraud detection stats
    fraud_count = sum(1 for r in llm_results.values() if r['is_fraud'])
    print(f"\nFraud Detection:")
    print(f"  Confirmed fraud: {fraud_count} / {len(llm_results)} ({fraud_count/len(llm_results)*100:.1f}%)")
    print(f"  Confirmed benign: {len(llm_results) - fraud_count}")
    
    # Fraud type distribution
    fraud_types_found = [r['fraud_type'] for r in llm_results.values() if r['is_fraud'] and r['fraud_type']]
    fraud_type_counts = Counter(fraud_types_found)
    
    print(f"\nFraud Type Distribution:")
    for fraud_type, count in fraud_type_counts.most_common():
        pct = count / len(fraud_types_found) * 100 if fraud_types_found else 0
        print(f"  {fraud_type}: {count} ({pct:.1f}%)")
    
    # Confidence and agreement
    confidences = [r['avg_confidence'] for r in llm_results.values()]
    agreements = [r['agreement_rate'] for r in llm_results.values()]
    
    print(f"\nQuality Metrics:")
    print(f"  Average confidence: {np.mean(confidences):.3f}")
    print(f"  Average agreement: {np.mean(agreements):.3f}")
    print(f"  Min agreement: {np.min(agreements):.3f}")
    print(f"  Max agreement: {np.max(agreements):.3f}")
    
    # Show sample explanations
    print("\n" + "="*70)
    print("SAMPLE EXPLANATIONS")
    print("="*70)
    
    # Show 3 examples
    sample_count = min(3, len(llm_results))
    sample_keys = list(llm_results.keys())[:sample_count]
    
    for i, idx in enumerate(sample_keys, 1):
        result = llm_results[idx]
        
        print(f"\n{'─'*70}")
        print(f"Example {i}:")
        print(f"{'─'*70}")
        print(f"Node ID: {result['node_id']}")
        print(f"Anomaly Score: {result['anomaly_score']:.4f}")
        print(f"\nLLM Analysis:")
        print(f"  Is Fraud: {result['is_fraud']}")
        print(f"  Fraud Type: {result['fraud_type']}")
        print(f"  Confidence: {result['avg_confidence']:.2f}")
        print(f"  Agreement: {result['agreement_rate']:.2%}")
        print(f"\nExplanation:")
        print(f"  {result['samples'][0]['explanation']}")
        print(f"\nEvidence:")
        print(f"  Features: {result['samples'][0]['evidence']['features']}")
        print(f"  Behaviors: {result['samples'][0]['evidence']['behaviors']}")
    
    # Final summary
    print("\n" + "="*70)
    print("[COMPLETE] DAY 3 COMPLETE!")
    print("="*70)
    
    print(f"\nGenerated files:")
    print(f"  [SAVED] ../results/day3_llm_results.pkl")
    
    print(f"\nWhat we accomplished:")
    print(f"  • Converted {len(llm_results)} technical explanations to natural language")
    print(f"  • Identified {fraud_count} fraudulent wallets")
    print(f"  • Categorized fraud types: {len(fraud_type_counts)} types found")
    print(f"  • Built consensus from {len(llm_results) * 5} LLM samples")
    print(f"  • Total cost: ${total_cost:.2f}")
    
    print(f"\nNext steps:")
    print(f"  - Day 4: Analysis & Visualization")
    print(f"  - Day 5: Documentation & Presentation")
    
    print("="*70)

else:
    print("\n[ERROR] No LLM results generated")
    print("Check for errors above")


SAVING RESULTS

✓ Saved 50 LLM results to:
  ../results/day3_llm_results.pkl
  File size: 137.4 KB

ANALYSIS

🔍 Fraud Detection:
  Confirmed fraud: 16 / 50 (32.0%)
  Confirmed benign: 34

📊 Fraud Type Distribution:
  Money laundering: 16 (100.0%)

📈 Quality Metrics:
  Average confidence: 0.336
  Average agreement: 0.888
  Min agreement: 0.600
  Max agreement: 1.000

SAMPLE EXPLANATIONS

──────────────────────────────────────────────────────────────────────
Example 1:
──────────────────────────────────────────────────────────────────────
Node ID: 339576533
Anomaly Score: 0.8104

🤖 LLM Analysis:
  Is Fraud: True
  Fraud Type: Money laundering
  Confidence: 0.69
  Agreement: 100.00%

📝 Explanation:
  The wallet shows a high anomaly score of 0.8104, indicating suspicious activity. The feature values, while not extremely high, suggest unusual transaction patterns that could align with money laundering behaviors. The low feature importance weights indicate that the model has limited confide